# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Danishh-ux/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

My lane: **Refresh / Content Opportunity Scoring**.

I'm choosing this because the starter notebooks (01, 02) already showed the exact shape of this problem: pages that are stale (not updated recently) and still visible (getting impressions) are the natural candidates for a review queue. This lane has a default dataset I already have access to (the starter CSV, with warehouse support later), and its output — a ranked queue with scores and reason codes — is a concrete, explainable deliverable rather than an abstract score. I can start immediately with data I've already loaded, and layer the warehouse's `fact_content_daily_performance` table on top later for time-aware validation.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# (No code needed for section 1 — lane choice is a text decision.)


## 2. The question: decision, action, cost of a wrong call

**Decision this improves:** which stale pages a content team should review first, out of thousands, given limited reviewer time each week.

**Who acts on it:** a content/SEO reviewer or team lead, who pulls the top N pages from the ranked queue and decides whether to refresh, redirect, or leave each one.

**Cost of a wrong recommendation:**
- False positive (flagged page didn't actually need review): wastes reviewer time — low-to-moderate cost, recoverable.
- False negative (a declining, high-traffic page never surfaces): the page keeps losing visibility unnoticed — higher cost, since impressions/traffic keep eroding before anyone looks at it.

Given that asymmetry, the model should lean toward higher recall on high-impression pages even if it means a few extra false positives in the queue.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# (No code needed for section 2 — framing is a text decision.)


## 3. Quick look at the data (2-3 real numbers)

Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) in ("notebooks", "work"):
    # in case this notebook is run from work/notebooks/, step back to repo root
    while not os.path.isdir("data") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: how much of the dataset is even "stale + visible" candidates
# Thresholds set from the data's own quartiles: days_since_last_update p75 ~= 104,
# impressions_90d p75 ~= 3615 -- i.e. "stale" and "visible" both mean top quartile.
stale_thresh = df["days_since_last_update"].quantile(0.75)
visible_thresh = df["impressions_90d"].quantile(0.75)
stale_visible = df[(df["days_since_last_update"] > stale_thresh) & (df["impressions_90d"] > visible_thresh)]
print(f"Pages stale (>{stale_thresh:.0f}d, top quartile) AND still visible "
      f"(>{visible_thresh:.0f} impressions/90d, top quartile): "
      f"{len(stale_visible)} of {len(df)} ({len(stale_visible)/len(df)*100:.1f}%)")

# Number 2: how much exposure is sitting in the declining bucket
declining = df[df["trend_direction"].str.lower() == "down"]
print(f"Declining pages account for {declining['impressions_90d'].sum():,} total "
      f"impressions_90d out of {df['impressions_90d'].sum():,} "
      f"({declining['impressions_90d'].sum()/df['impressions_90d'].sum()*100:.1f}% of all exposure)")

# Number 3: precision already achievable, from notebook 02 -- hand rule vs tree
print("From notebook 02: a simple hand rule already hits roughly 0.68-0.90 "
      "Precision@20-50 on this exact staleness x visibility framing, showing "
      "there is real learnable signal here worth extending.")


Pages stale (>104d, top quartile) AND still visible (>3615 impressions/90d, top quartile): 42 of 30000 (0.1%)
Declining pages account for 79,994,363 total impressions_90d out of 156,010,989 (51.3% of all exposure)
From notebook 02: a simple hand rule already hits roughly 0.68-0.90 Precision@20-50 on this exact staleness x visibility framing, showing there is real learnable signal here worth extending.


## 4. Careful words: what I can and can't claim

**What I can claim:** observed patterns in this snapshot (e.g. "X% of pages are stale and still visible"), directional signal about which features associate with decline, and decision-support rankings — a prioritized list, not a verdict.

**What I cannot claim:** that refreshing a page *causes* recovery (that needs a real experiment, not this observational data), that I've reverse-engineered any part of Google's ranking algorithm, or that any single score is "the truth" rather than one useful signal among several. Precalculated fields like `trend_direction` will be used as label/context, never re-fed as a feature (that's leakage, as shown in notebook 02, section 3). Any claim I make will use words like "observed," "associated with," or "directional" — never "proves" or "predicts the algorithm."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# (No code needed for section 4 — this is a statement of scope.)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.